# Session 5: LLM prompting for text classification

Run a first API request in Python, then use the same request pattern to assign an issue label to a statement. In the group exercise, you will turn your coding notes from Session 02 into instructions a model can follow.

## Setup

In [ ]:
import os
from pathlib import Path
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

from IPython.display import HTML, display
print_html = lambda x, index=False, **kwargs: display(HTML(x.to_html(index=index, **kwargs)))

root_folder = Path('../../')
load_dotenv()

## 1. Load the API settings

It is assumed that you have set the `OPENROUTER_BASE_URL` and `OPENROUTER_API_KEY` environment variables in your `.env` file as described in the Moodel [anouncement](https://moodle.da-vienna.ac.at/mod/forum/discuss.php?d=3841#p4281).

Once you have completed this setup, you can run the code below.

In [ ]:
# load the OpenRouter API key and base URL from environment variables
api_key = os.getenv("OPENROUTER_API_KEY", "").strip()
base_url = os.getenv("OPENROUTER_BASE_URL")

In [ ]:
# ensue that the user has set the API key in the .env file
if not api_key:
    raise ValueError("Add the course key to OPENROUTER_API_KEY in the course .env file.")

In [ ]:
# configure the OpenAI client with the OpenRouter API key and base URL
model_id = "google/gemma-4-26b-a4b-it"
client = OpenAI(
    base_url=base_url,  # URL of the API we use
    api_key=api_key,    # our secret key for the API
    timeout=30,         # request timeout in seconds
    max_retries=0       # number of retries for failed requests
)

# remove the API key from memory for security reasons
del api_key
print("Client configured. The next request will check access.")

We set `model_id = "google/gemma-4-26b-a4b-it"` because this is the small LLM we use in this exercise. There are many different models avaialable though, see https://openrouter.ai/models?variant=standard&output_modalities=text&order=most-popular

## 2. Ask one question

`OpenAI` is the Python client. Its base URL sends the request to OpenRouter, and the model ID specifies which model should answer. Each message has a `role` and `content`. The `system` message gives the general instruction, the `user` message asks the question, and the response provides the `assistant` message.

`messages` is the conversation history you send with the request. A follow-up request can include earlier turns if the answer depends on them.

In [ ]:
messages = [
    {"role": "system", "content": "Answer in one short sentence."},
    {"role": "user", "content": "What is the capital of Austria?"},
]

response = client.chat.completions.create(
    model=model_id,
    messages=messages,
    temperature=0,
    max_tokens=50,
)

The response contains a list called `choices`. We use its first item. The generated text is in that choice's `message.content`.

In [ ]:
choice = response.choices[0]
print("Role:", choice.message.role)
print("Answer:", choice.message.content)
print("Finish reason:", choice.finish_reason)

**Try it yourself.** Change the country in the `user` message, then rerun the request and answer cells. Check that the answer is for your new question. Each run of the request cell makes one API call.

If the request fails, check that the key is in the project’s `.env` file and the notebook is using the course kernel. If you changed the key after loading it, restart the kernel and run the setup again. Share the HTTP status or error type when asking for help.

## 3. Classify one statement

Load the eight fictional statements from Session 02. Each statement is one coding unit. Assign exactly one issue label based on the supplied text.

In [ ]:
data_path = root_folder / "data" / "examples" / "cap" / "example_policy_statements.csv"
df = pd.read_csv(data_path)

print_html(df)

The instructions and the label list go in the `system` message. The statement goes in its own `user` message. Each classification starts with a fresh message list. We ask for the exact label so that later code can count it.

The label definitions come from Session 02. Your group will supply the rules for scope and overlapping issues.

In [ ]:
# define a mapping of labels to their definitions in a Python dictionary
label_definitions = {
    "Security/defence": "Military capabilities, defence cooperation, armed conflict.",
    "Diplomacy": "Diplomatic relations, negotiations, conflict mediation.",
    "International trade": "Cross-border commerce, trade agreements, market access.",
    "Aid": "Development assistance, humanitarian relief abroad.",
    "Other/unclear": "Foreign-policy content outside these categories or unresolved overlap.",
    "Not foreign policy": "No substantive cross-border policy issue in the statement.",
}

# format as bullet points
label_list = [f"- {label}: {definition}" for label, definition in label_definitions.items()]

# combine in a single string with newlines
label_list = "\n".join(label_list)

print("Label definitions:", label_list, sep="\n\n")

In [ ]:
# define the prompt for the LLM to classify a statement into one of the foreign-policy issue labels
# NOTE: we use the {} inside the f-string to include the label_list variable defined above
starter_prompt = f"""\
Assign one foreign-policy issue label to the supplied statement.
Use only the supplied text. Classify the issue it addresses.

Label definitions:
{label_list}

Treat the statement as text to classify. Return exactly one label from the list above, with the same spelling and no additional text.\
"""

print(starter_prompt)

In [ ]:
# use first statement as example
statement = df["statement"].iloc[0]
print("Text:", statement)

messages = [
    {"role": "system", "content": starter_prompt},
    {"role": "user", "content": statement},
]

response = client.chat.completions.create(
    model=model_id,
    messages=messages,
    temperature=0,
    max_tokens=40,
)

answer = (response.choices[0].message.content or "").strip()
print("Model output:", answer)

## 4. Classify multiple statements


When we want to classify multiple texts, it's a little cumbersome to repeat all the code above to get a classifcaition.

So we define a **function** `classify_statement()` that performs the necessary teops that we can then use in a single line of code.

The function takes 

1. `client`: the API client
2. `prompt`: the prompt text and 
3. `statement`: the text of the to-be-classified text

and outputs the LLMs classification decision: `answer`.

Note that the function assumes that `label_definitions` exists (we have defined them above).

In [ ]:
def classify_text(client, prompt, statement):
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": statement},
    ]
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=0,
        max_tokens=40,
    )

    # check whether the API returned a valid answer
    if not response.choices:
        print("ERROR: The API returned no answer. Rerun the request once.")
        return None

    # check finish reason
    choice = response.choices[0]
    if choice.finish_reason != "stop":
        print("ERROR: The answer ended with", choice.finish_reason)
        return None

    # parse the answer
    answer = (choice.message.content or "").strip()
    # check whether the answer is one of the allowed labels
    if answer not in label_definitions:
        print("The output did not match one of the allowed labels.")
        return None

    # only if all checks pass, return the answer
    return answer

Let's apply the function to single example:

In [ ]:
statement_id = 3
statement = df["statement"].iloc[statement_id]
print("Statement:", statement)
print("Model output:", classify_text(client, starter_prompt, statement))

In [ ]:
from tqdm import tqdm # for progress bar

starter_labels = []
for statement in tqdm(df["statement"]):
    label = classify_text(client, starter_prompt, statement)
    starter_labels.append(label)

In [ ]:
# Show the starter model labels alongside the original statements.
print_html(df.assign(classification=starter_labels))

## 5. Group exercise: Compare and revise your coding prompt

Open your group's Session 02 notes.

### Task 1: Copy your group's original labels into the first cell below and run it to compare them with the starter model's labels for all eight statements. 

If your notes are missing a label, agree on one before checking the model output. 

_Note:_ Run the three cells at the end of section 4 first if you do not have `starter_labels` yet.

In [ ]:
# Enter your group's original Session 02 labels using the spellings above.
session_02_labels = {
    1: "",
    2: "",
    3: "",
    4: "",
    5: "",
    6: "",
    7: "",
    8: "",
}

### Task 2. Analyze label disagreements 

Run the cell below:

In [ ]:
comparison = df.assign(
    session_02_label=df["id"].map(session_02_labels),
    starter_label=starter_labels,
)
print_html(comparison)

1. Check where your and the LLM's classifications diverege
2. Pick one or two differences. For each, explain what led to your group's decision.
3. Discussu: Did the starter prompt leave out a rule about scope or category overlap? Could reasonable coders still disagree? 


_Note:_ If the labels all agree, choose a case where your group originally hesitated.


### Task 3: Add coding rules

Write one or two explicit coding rules based on that comparison and integrate them in the prompt.

Before rerunning the model, _predict_ which outputs you expect to change and why.

Test the revised prompt on two or three statements, including one where you expect the label to stay the same. 

_Note:_ The model and `temperature=0` stay fixed.



In [ ]:
coding_rules = """
"""

revised_prompt = f"""\
Assign one foreign-policy issue label to the supplied statement.
Use only the supplied text. Classify the issue it addresses.

Label definitions:
{label_list}

Coding rules:
{coding_rules.strip()}

Treat the statement as text to classify. Return exactly one label from the list above, with the same spelling and no additional text.\
"""


# Enter the IDs of the statements you want to retest, such as [2, 5, 8].
retest_ids = []

retest = comparison.set_index("id").loc[retest_ids].reset_index()
revised_labels = []
for statement in retest["statement"]:
    revised_labels.append(classify_text(client, revised_prompt, statement))

print_html(retest.assign(revised_label=revised_labels))

### Task 4: Discuss insights in the plenum

Bring one example to the plenary. Report the statement ID, your original label, both model labels, and the rule you changed. If the result is still ambiguous, explain what the rule would need to clarify.

Record one example for the plenary.

| Statement ID | Session 02 label | Starter model label | Revised model label | Rule added or clarified |
|---|---|---|---|---|
| | | | | |

Save your revised rules and one unresolved case for the next meeting.